# 🏭 Full 550-Stock Pipeline (walk-forward)
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Realosunboy6/550-Stocks-Portfolio-Theory-Python-Julia-/blob/main/notebooks/07_full_550_pipeline.ipynb)

The original `Portfolio_Optimization_COLAB.ipynb` 8-phase pipeline, re-built on `portlab` in a fraction of the code: universe download → Ledoit-Wolf → 6 optimizer strategies → **rolling walk-forward backtest** (252d train / 21d rebalance, transaction costs, no look-ahead) → CVaR + stress tests → final dashboard.

In [ ]:
#@title Setup — run this first {display-mode: "form"}
try:
    import portlab
except ImportError:
    %pip install -q "portlab @ git+https://github.com/Realosunboy6/550-Stocks-Portfolio-Theory-Python-Julia-.git"
    import portlab
print("portlab", portlab.__version__, "ready")

In [ ]:
#@title Settings {display-mode: "form"}
universe = "sector_etfs_plus_bonds"  #@param ["sector_etfs_plus_bonds", "tech_sector_stocks", "full_550_stocks"]
start_date = "2019-01-01"  #@param {type:"date"}
train_days = 252           #@param {type:"number"}
rebalance_days = 21        #@param {type:"number"}
transaction_cost_bps = 10  #@param {type:"number"}
max_weight = 0.10          #@param {type:"number"}
SMOKE = False

In [ ]:
from portlab.data import SECTOR_ETFS, get_returns, sector_tickers, all_stocks

if universe == "sector_etfs_plus_bonds":
    ticks = list(SECTOR_ETFS.values()) + ["TLT", "IEF", "GLD"]
elif universe == "tech_sector_stocks":
    ticks = sector_tickers("Information Technology")
else:
    ticks = all_stocks()
if SMOKE:
    ticks = ticks[:8]
rets = get_returns(ticks, start_date)
print(f"universe: {rets.shape[1]} assets, {rets.shape[0]} days")

In [ ]:
import pandas as pd
from portlab import optimize as opt
from portlab.backtest import RollingBacktest, equal_weight
from portlab.optimize import mean_cov

bounds = (0.0, max_weight)

def _mv(fn):
    def strat(train):
        mu, cov = mean_cov(train)
        return fn(mu, cov)
    return strat

strategies = {
    "Equal Weight": equal_weight,
    "GMV (LW)": _mv(lambda m, c: opt.gmv(m, c, bounds=bounds)),
    "Max Sharpe (LW)": _mv(lambda m, c: opt.max_sharpe(m, c, bounds=bounds)),
    "Risk Parity": _mv(lambda m, c: opt.equal_risk_contribution(c, bounds=bounds)),
    "Min CVaR": lambda train: opt.min_cvar(train, bounds=bounds),
    "Inverse Vol": _mv(lambda m, c: opt.inverse_vol(c)),
}
rb = RollingBacktest(rets, train_window=train_days, rebalance_every=rebalance_days,
                     tc_bps=transaction_cost_bps)
oos, summary = rb.run_many(strategies)
summary.style.format("{:.3f}")

In [ ]:
from portlab import plots
plots.growth_chart(oos.fillna(0), initial=10000).show()
plots.drawdown_chart(oos.fillna(0)).show()

In [ ]:
# Stress tests on the best out-of-sample strategy
from portlab.backtest import episode_returns, worst_windows
best = summary.loc["Sharpe Ratio"].astype(float).idxmax()
print("Best OOS strategy:", best)
display(worst_windows(oos[best].dropna(), window=21))
display(episode_returns(oos[best].dropna()))

In [ ]:
plots.corr_heatmap(oos.corr(), "Strategy Return Correlations").show()